# Predição: Imagens da USP com o modelo MSO-V1

Autor:  Viviane da Rosa Sommer

Atualizado: 19/12/2024

## Objetivo:
Notebook para fazer a predição das imagens da USP pelo modelo MSO-V1, para analisar o desempenho do mesmo.

In [ ]:
!ruff check prediction_model-usp.ipynb

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
import glob
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os 
import torchvision
import torch
from tqdm import tqdm
import random

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
RESIZED_DIM_CORAL = 800
BATCH_SIZE = 50
MAX_SIZE = (600, 600)
INPUT_DIRECTORY = "datasets/Terceiros-USP"
OUTPUT_DIRECTORY = "results-usp"
coral_model = YOLO('runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> tuple:
    """
    Reads an image from a file and returns it along with its dimensions.

    Parameters:
        filename (str): Path to the image file.

    Returns:
        tuple: A tuple containing:
            - img (np.ndarray): The loaded image in BGR format.
            - base_height (int): Height of the image.
            - base_width (int): Width of the image.
            If the image cannot be loaded, returns (None, None, None).
    """
    img = cv2.imread(filename)
    if img is None:
        print(f"Failed to load image: {filename}")
        return None, None, None
        
    base_height, base_width = img.shape[:2]
    return img, base_height, base_width


def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.array) -> tuple:
    """
    Predicts coral objects using the coral model and returns a binary mask for coral regions.

    Parameters:
        img (np.array): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - centered_mask (np.ndarray): Resized coral mask with centered annotations.
            - torch.Tensor: The centered mask as a PyTorch tensor.
    """
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            final_coral_mask = torch.any(coral_masks, dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    
    final_coral_mask_np = final_coral_mask.cpu().numpy()
    mask_height, mask_width = final_coral_mask_np.shape

    resized_final_coral_mask = cv2.resize(final_coral_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    centered_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    y_offset = (original_height - resized_final_coral_mask.shape[0]) // 2
    x_offset = (original_width - resized_final_coral_mask.shape[1]) // 2
    if (y_offset >= 0 and x_offset >= 0 and 
        y_offset + resized_final_coral_mask.shape[0] <= original_height and 
        x_offset + resized_final_coral_mask.shape[1] <= original_width):
        
        centered_mask[y_offset:y_offset + resized_final_coral_mask.shape[0], 
                      x_offset:x_offset + resized_final_coral_mask.shape[1]] = resized_final_coral_mask
    else:
        centered_mask = resized_final_coral_mask.copy()

    return centered_mask, torch.from_numpy(centered_mask).int()


def save_image(img: np.ndarray, 
               intersection_mask: np.ndarray,
               file_base_name: str) -> None:
    """
    Displays and saves annotated images, highlighting intersections between model and expert annotations.

    Parameters:
    -----------
    img : np.ndarray
        The original image in BGR format.
    intersection_mask : np.ndarray
        Binary mask indicating intersections between model and expert annotations.
    file_base_name : str
        Base name for the output files.

    This function performs the following steps:
    1. Creates a copy of the original image for annotation.
    2. Draws contours on the annotated image where there are intersections in the intersection mask.
    3. Saves the annotated image as a PNG file in the specified output directory with a filename based on the input file base name.
    4. Optionally displays the annotated image using Matplotlib.

    The annotated image highlights:
    - **Model Annotation**: Contours detected by the model are displayed in red, with intersections overlaid on the original image.

    The output file is saved in the `OUTPUT_DIRECTORY` with the specified base name.
    """
    
    annotated_img = img.copy()

    if intersection_mask.any():
        contours, _ = cv2.findContours(intersection_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(annotated_img, contours, -1, (0, 0, 255), thickness=5) 

    output_file_path = os.path.join(OUTPUT_DIRECTORY, f"{file_base_name}.png")
    cv2.imwrite(output_file_path, annotated_img)

    plt.figure(figsize=(10, 7))
    plt.title('Model Annotation')
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.tight_layout()
    plt.close()
    

def process_file(filename: str) -> None:
    """
    Processes a single image file.

    Args:
        filename (str): Path to the image file to process.

    Returns:
        None
    """
    try:
        if filename.split("/")[-1].split(".")[-1] in ["jpg", "png", "JPG"]:
            img, base_height, base_width = read_image(filename)
            if img is None:
                return

            coral_mask_resized, coral_mask_tensor = prediction_coral(img)
            save_image(img, coral_mask_resized, filename.split("/")[-1])
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        

## Processamento das imagens

In [ ]:
all_images_input = glob.glob(f"{INPUT_DIRECTORY}/**")

for file in tqdm(all_images_input, desc='Processando imagens',unit='arquivo', ncols=80, bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}'):
    process_file(file)

## Plot de 40% das imagens

In [ ]:
all_images_output = glob.glob(f"{OUTPUT_DIRECTORY}/**")
sample_images_output = random.sample(all_images_output, int(len(all_images_output) * 0.4))

for img_path in sample_images_output:
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure()
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(os.path.basename(img_path))
    plt.show()

In [ ]:
jupyter nbconvert --to html prediction_model-usp.ipynb